# Step 4 — ML Classifier

**Goal:** Train machine learning classifiers to automatically detect othering language.

We compare two feature strategies:
1. **TF-IDF** — fast, sparse bag-of-words + bigrams (baseline)
2. **Sentence embeddings** — dense semantic vectors via `all-MiniLM-L6-v2` (better)

And two model families:
- **Logistic Regression** — interpretable, calibrated probabilities
- **LinearSVC** — often stronger on text classification tasks

**Input:** `data/processed/dataset_othering.csv`  
**Output:** `src/othering_classifier.pkl` — best model for reuse in later steps

In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

from classifier import (
    build_tfidf_features,
    build_embedding_features,
    train_and_evaluate,
    save_model,
    load_model,
)

MAX_ROWS        = None   # set to e.g. 5000 for faster testing
SKIP_EMBEDDINGS = False  # set True to skip sentence-transformer step

## Task 1 — Load dataset

In [ ]:
df = pd.read_csv('../data/processed/dataset_othering.csv')
print(f'Loaded {len(df):,} rows')

if MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=42).reset_index(drop=True)
    print(f'Subsampled to {len(df):,} rows')

text_col = 'clean_text' if 'clean_text' in df.columns else 'text'
texts  = df[text_col].fillna('').tolist()
labels = df['has_othering'].astype(int).tolist()

pos = sum(labels)
print(f'Othering: {pos:,} ({pos/len(labels)*100:.1f}%)  Non-othering: {len(labels)-pos:,}')

# Stratified 80/20 split
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
y_train_arr = np.array(y_train, dtype=bool)
y_test_arr  = np.array(y_test,  dtype=bool)
print(f'Train: {len(y_train):,}  Test: {len(y_test):,}')

## Task 2 — TF-IDF features

In [ ]:
X_tfidf_tr, vect = build_tfidf_features(X_train_txt)
X_tfidf_te       = vect.transform(X_test_txt)
print(f'TF-IDF matrix: train {X_tfidf_tr.shape}, test {X_tfidf_te.shape}')

tfidf_res = train_and_evaluate(
    X_tfidf_tr, X_tfidf_te,
    y_train_arr, y_test_arr,
    feature_name='TF-IDF',
    test_texts=X_test_txt,
)

In [ ]:
# Plot confusion matrices for TF-IDF models
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, r in zip(axes, tfidf_res['results']):
    disp = ConfusionMatrixDisplay(r['cm'], display_labels=['no_othering', 'othering'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f"{r['model_name']} — TF-IDF")
plt.tight_layout()
plt.savefig('../reports/figures/cm_tfidf.png', dpi=120)
plt.show()

## Task 3 — Sentence-transformer embeddings

In [ ]:
if not SKIP_EMBEDDINGS:
    print('Encoding train set…')
    X_emb_tr = build_embedding_features(X_train_txt)
    print('Encoding test set…')
    X_emb_te = build_embedding_features(X_test_txt)
    print(f'Embedding shape: {X_emb_tr.shape}')

    emb_res = train_and_evaluate(
        X_emb_tr, X_emb_te,
        y_train_arr, y_test_arr,
        feature_name='Embeddings',
        test_texts=X_test_txt,
    )
else:
    emb_res = {'results': []}
    print('Skipped (SKIP_EMBEDDINGS=True)')

In [ ]:
if emb_res['results']:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, r in zip(axes, emb_res['results']):
        disp = ConfusionMatrixDisplay(r['cm'], display_labels=['no_othering', 'othering'])
        disp.plot(ax=ax, colorbar=False)
        ax.set_title(f"{r['model_name']} — Embeddings")
    plt.tight_layout()
    plt.savefig('../reports/figures/cm_embeddings.png', dpi=120)
    plt.show()

## Task 4 — Model comparison

In [ ]:
all_results = tfidf_res['results'] + emb_res['results']

summary = pd.DataFrame([{
    'model':     r['model_name'],
    'features':  r['feature'],
    'precision': round(r['precision'], 4),
    'recall':    round(r['recall'],    4),
    'f1':        round(r['f1'],        4),
} for r in all_results]).sort_values('f1', ascending=False)

display(summary)

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(summary))
labels_bar = [f"{r['model']}\n{r['features']}" for _, r in summary.iterrows()]
ax.bar(x, summary['f1'], color=['#2196F3','#FF9800','#4CAF50','#E91E63'][:len(summary)])
ax.set_xticks(x)
ax.set_xticklabels(labels_bar, fontsize=9)
ax.set_ylabel('F1 score')
ax.set_title('Model comparison — F1 on othering detection')
ax.set_ylim(0, 1)
for i, (_, row) in enumerate(summary.iterrows()):
    ax.text(i, row['f1'] + 0.01, f"{row['f1']:.3f}", ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=120)
plt.show()

## Task 5 — Save best model

In [ ]:
best = max(all_results, key=lambda r: r['f1'])
print(f"Best model: {best['model_name']} ({best['feature']})  F1={best['f1']:.4f}")

payload = {
    'model':      best['model'],
    'vectorizer': vect if best['feature'] == 'TF-IDF' else None,
    'model_name': f"{best['model_name']}_{best['feature']}",
}
save_model(payload, '../src/othering_classifier.pkl')

## Task 6 — Full report on best model

In [ ]:
print(best['report'])

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(best['cm'], display_labels=['no_othering', 'othering'])
disp.plot(ax=ax, colorbar=False)
ax.set_title(f"Best model: {best['model_name']} ({best['feature']})")
plt.tight_layout()
plt.savefig('../reports/figures/cm_best.png', dpi=120)
plt.show()